In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()

for candidate in [PROJECT_ROOT, PROJECT_ROOT.parent]:
    if (candidate / "pyproject.toml").exists():
        PROJECT_ROOT = candidate
        break

SRC_DIR = PROJECT_ROOT / "src"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

if SRC_DIR.exists() and str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw dir: {RAW_DIR}")
print(f"Processed dir: {PROCESSED_DIR}")

Project root: /Users/rodrigue.lawson/VSCode Projects/lexcare-ai
Raw dir: /Users/rodrigue.lawson/VSCode Projects/lexcare-ai/data/raw
Processed dir: /Users/rodrigue.lawson/VSCode Projects/lexcare-ai/data/processed


In [2]:
import pandas as pd

from app.infrastructure.pdf_loader import PDFLoader
from app.repositories.document_repository import FileDocumentRepository
from app.services.document_processing_service import DocumentProcessingService
from app.services.chunking_service import ChunkingService
from app.services.text_cleaning_service import TextCleaningService

In [3]:
pdf_files = sorted((RAW_DIR / "gesetze_im_internet").rglob("*.pdf"))

print(f"Found {len(pdf_files)} PDFs:")
for pdf in pdf_files:
    print("-", pdf.name)

Found 3 PDFs:
- PUEG_RefE_Pflegereform_bf.pdf
- SGB_11.pdf
- SGB_5.pdf


In [4]:
loader = PDFLoader()
repo = FileDocumentRepository()
cleaner = TextCleaningService()

In [5]:
def run_chunking_test(pdf_path, chunk_size: int, chunk_overlap: int):
    loaded = loader.load(
        str(pdf_path),
        source="gesetze-im-internet",
        document_type="law",
        topic="healthcare-law",
    )

    saved = repo.save(loaded)

    processor = DocumentProcessingService(
        cleaning_service=cleaner,
        chunking_service=ChunkingService(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
        ),
    )

    chunks = processor.process(saved)

    return {
        "file_name": pdf_path.name,
        "chunk_size": chunk_size,
        "chunk_overlap": chunk_overlap,
        "page_count": loaded.metadata.extra.get("page_count"),
        "raw_text_length": len(loaded.text),
        "clean_text_length": len(cleaner.clean(loaded.text)),
        "chunk_count": len(chunks),
        "avg_chunk_length": round(sum(len(c.text) for c in chunks) / len(chunks), 2) if chunks else 0,
        "first_chunk_preview": chunks[0].text[:300] if chunks else "",
        "last_chunk_preview": chunks[-1].text[:300] if chunks else "",
    }

In [6]:
configs = [
    (800, 100),
    (1000, 200),
    (1200, 200),
    (1500, 250),
]

results = []

for pdf_path in pdf_files:
    for chunk_size, chunk_overlap in configs:
        results.append(run_chunking_test(pdf_path, chunk_size, chunk_overlap))

df = pd.DataFrame(results)
df

,file_name,chunk_size,chunk_overlap,page_count,raw_text_length,clean_text_length,chunk_count,avg_chunk_length,first_chunk_preview,last_chunk_preview
0,PUEG_RefE_Pflegereform_bf.pdf,800,100,112,359167,355150,508,798.67,Referentenentwurf\ndes Bundesministeriums für ...,htungen und um die Ausweitung der Inanspruchna...
1,PUEG_RefE_Pflegereform_bf.pdf,1000,200,112,359167,355150,444,999.17,Referentenentwurf\ndes Bundesministeriums für ...,Änderungen treten mit Wirkung zum 1. Januar 20...
2,PUEG_RefE_Pflegereform_bf.pdf,1200,200,112,359167,355150,355,1199.61,Referentenentwurf\ndes Bundesministeriums für ...,ukturierung verbundenen redaktionellen Folgeän...
3,PUEG_RefE_Pflegereform_bf.pdf,1500,250,112,359167,355150,284,1499.37,Referentenentwurf\ndes Bundesministeriums für ...,Feststellung der Pflegebedürftigkeit be\n-\ntr...
4,SGB_11.pdf,800,100,177,785894,784738,1121,799.69,Ein Service des Bundesministerium der Justiz u...,ersorgung 40 %\n0 10 20 30 40 Gewichtete\nPunk...
5,SGB_11.pdf,1000,200,177,785894,784738,981,999.49,Ein Service des Bundesministerium der Justiz u...,ersorgung 40 %\n0 10 20 30 40 Gewichtete\nPunk...
6,SGB_11.pdf,1200,200,177,785894,784738,785,1199.15,Ein Service des Bundesministerium der Justiz u...,ersorgung 40 %\n0 10 20 30 40 Gewichtete\nPunk...
7,SGB_11.pdf,1500,250,177,785894,784738,628,1498.91,Ein Service des Bundesministerium der Justiz u...,oblemlagen\n0 1 – 2 3 – 4 5 – 6 7 – 65 Summe d...
8,SGB_5.pdf,800,100,568,2662222,2658704,3799,799.54,Ein Service des Bundesministerium der Justiz u...,24 I Nr. 101 mWv 26.3.2024 aufgrund textlicher...
9,SGB_5.pdf,1000,200,568,2662222,2658704,3324,999.52,Ein Service des Bundesministerium der Justiz u...,Prüfung der festgelegten Maßnahmen besteht.\n\...


In [7]:
df[[
    "file_name",
    "chunk_size",
    "chunk_overlap",
    "page_count",
    "chunk_count",
    "avg_chunk_length",
]]

,file_name,chunk_size,chunk_overlap,page_count,chunk_count,avg_chunk_length
0,PUEG_RefE_Pflegereform_bf.pdf,800,100,112,508,798.67
1,PUEG_RefE_Pflegereform_bf.pdf,1000,200,112,444,999.17
2,PUEG_RefE_Pflegereform_bf.pdf,1200,200,112,355,1199.61
3,PUEG_RefE_Pflegereform_bf.pdf,1500,250,112,284,1499.37
4,SGB_11.pdf,800,100,177,1121,799.69
5,SGB_11.pdf,1000,200,177,981,999.49
6,SGB_11.pdf,1200,200,177,785,1199.15
7,SGB_11.pdf,1500,250,177,628,1498.91
8,SGB_5.pdf,800,100,568,3799,799.54
9,SGB_5.pdf,1000,200,568,3324,999.52


In [8]:
pivot = df.pivot_table(
    index=["file_name"],
    columns=["chunk_size", "chunk_overlap"],
    values="chunk_count",
    aggfunc="first",
)

pivot

chunk_size,800,1000,1200,1500
chunk_overlap,100,200,200,250
file_name,,,,
PUEG_RefE_Pflegereform_bf.pdf,508,444,355,284
SGB_11.pdf,1121,981,785,628
SGB_5.pdf,3799,3324,2659,2127


In [9]:
selected = df[(df["chunk_size"] == 1000) & (df["chunk_overlap"] == 200)]

selected[[
    "file_name",
    "chunk_count",
    "avg_chunk_length",
    "first_chunk_preview",
    "last_chunk_preview",
]]

,file_name,chunk_count,avg_chunk_length,first_chunk_preview,last_chunk_preview
1,PUEG_RefE_Pflegereform_bf.pdf,444,999.17,Referentenentwurf\ndes Bundesministeriums für ...,Änderungen treten mit Wirkung zum 1. Januar 20...
5,SGB_11.pdf,981,999.49,Ein Service des Bundesministerium der Justiz u...,ersorgung 40 %\n0 10 20 30 40 Gewichtete\nPunk...
9,SGB_5.pdf,3324,999.52,Ein Service des Bundesministerium der Justiz u...,Prüfung der festgelegten Maßnahmen besteht.\n\...


In [10]:
for pdf_path in pdf_files:
    loaded = loader.load(
        str(pdf_path),
        source="gesetze-im-internet",
        document_type="law",
        topic="healthcare-law",
    )

    saved = repo.save(loaded)

    processor = DocumentProcessingService(
        cleaning_service=cleaner,
        chunking_service=ChunkingService(chunk_size=1000, chunk_overlap=200),
    )

    chunks = processor.process(saved)

    print("=" * 100)
    print(pdf_path.name)
    print("Chunks:", len(chunks))
    print("First chunk:")
    print(chunks[0].text[:800] if chunks else "No chunks")
    print()

PUEG_RefE_Pflegereform_bf.pdf
Chunks: 444
First chunk:
Referentenentwurf
des Bundesministeriums für Gesundheit
Entwurf eines Gesetzes zur Unterstützung und Entlastung in der Pflege
(Pflegeunterstützungs- und -entlastungsgesetz – PUEG)
A. Problem und Ziel
Auf der Basis von im Koalitionsvertrag für diese Legislaturperiode vorgesehenen Maßnah -
men zur Verbesserung der Situation in der Pflege sollen Anpassungen in der Pflegeversi -
cherung vorgenommen werden. Insbesondere sollen die häusliche Pflege gestärkt und pfle-
gebedürftige Menschen und ihre Angehörigen sowie anderen Pflegepersonen entlastet, die
Arbeitsbedingungen für professionell Pflegende weiter verbessert sowie die Potentiale der
Digitalisierung für Pflegebedürftige und für Pflegende noch besser nutzbar gemacht wer -
den. Es soll eine automatische, regelhafte Anpassung der Geld- und S

SGB_11.pdf
Chunks: 981
First chunk:
Ein Service des Bundesministerium der Justiz und für Verbraucherschutz
sowie des Bundesamts für Justiz ‒ ww

In [11]:
output_path = PROCESSED_DIR / "chunking_experiments.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(output_path, index=False)
print(f"Saved to: {output_path}")

Saved to: /Users/rodrigue.lawson/VSCode Projects/lexcare-ai/data/processed/chunking_experiments.csv


In [12]:
sample_chunk = chunks[0]

print("Document ID:", sample_chunk.document_id)
print("Chunk ID:", sample_chunk.chunk_id)
print("Chunk Index:", sample_chunk.chunk_index)
print("Source:", sample_chunk.metadata.source)
print("Topic:", sample_chunk.metadata.topic)

Document ID: f7fc5ef1-75db-46e2-aaa0-b155c951f0ca
Chunk ID: f7fc5ef1-75db-46e2-aaa0-b155c951f0ca-chunk-0000
Chunk Index: 0
Source: gesetze-im-internet
Topic: healthcare-law
